<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="./img/btp-banner.gif" alt="BTP A&C">
</div>

**SAP-RPT-1 in Action** — Architecting Agentic Supply Chain on SAP Business AI Platform

# Exercise 1A — From Dark Data to Delay Prediction
## Use Case: Just-In-Time (JIT) Supply Chain Risk

You will predict which purchase orders are at risk of delay *before* they happen, using SAP's **sap-rpt-1** (Relational Pretrained Transformer) model on SAP AI Core — and turn that prediction into a governed business action.

### The Use Case: Intelligent JIT Risk Prediction

**BestRun Technologies** is a fast-growing, **fabless** maker of smart IoT security devices: it designs the products and relies on contract manufacturers, so factory line time is booked weeks ahead and expensive to lose. Its two most critical inputs — a secure-element chip and a MEMS sensor — are effectively **sole-sourced** on 30+ week lead times in an allocation-constrained market, and because the product refreshes on a short cycle, components go obsolete fast — so **holding a large safety-stock buffer isn't viable** (inventory that sits becomes a write-off). The result isn't JIT by choice so much as a **cornered position**: no buffer, no *pre-qualified* second source (switching is possible, but carries its own cost premium and lead-time risk), and a booked production line that idles the moment a critical shipment slips — a "line-down" situation costing **$15,000+ per hour** in idle labor and missed delivery penalties.

**The Problem:** Current supply chain management is reactive. Managers only realize a Purchase Order (PO) is delayed when the truck doesn't arrive. While S/4HANA contains years of historical supplier performance data, it remains "dark data" — underutilized for forward-looking insights.

**The Solution:** In this workshop, you will use SAP's **sap-rpt-1** (Relational Pretrained Transformer) model to predict which POs are at risk of delay *before* they happen, enabling proactive mitigation.

---

### How this exercise is structured

This notebook is built as **short, self-contained sections**. Each one follows the same rhythm so you always know where you are and what you just proved:

> 📘 **KNOWLEDGE POINT** — the idea &nbsp;→&nbsp; 🎯 **Outcome** — what you'll be able to do &nbsp;→&nbsp; ⌨️ **Your Turn** — you run it &nbsp;→&nbsp; ✓ **Checkpoint** — what you proved

You climb the **Decision Ladder** one rung at a time, and *feel* why each rung exists before you build it.

| # | Section | Rung | You'll prove |
|---|---------|------|--------------|
| **1** | Setup & Configuration | — | Live connection to SAP AI Core |
| **2** | The Data & the Dark-Data Problem | — | You can read the signal a model will learn |
| **3** | **Rung 1 — Rules Only (no AI)** | 🔵 Rules | *Why rules alone aren't enough* |
| **4** | **Rung 2 — Predict with SAP-RPT-1** | 🟢 Predict | Prediction catches what rules can't |
| **5** | Rules + Predict = Policy | 🔵+🟢 | A score only matters once it drives an action |
| **6** | Mitigation Proposal | 🔵+🟢 | The model proposes; the human decides |
| **7** | **Apply to YOUR Landscape** | — | You can place your own use case on the ladder |
| **8** | → From Workshop to Production | — | The questions that come next |

**You are here → Section 0: Orientation.**

## The Decision Ladder: Rules → Predict → Reason

Every automation choice in this workshop sits on a simple ladder. **Start low by default, and climb only as high as the decision actually requires.**

| Rung | Use when the decision is... | SAP capability | What you must govern |
|------|------------------------------|----------------|----------------------|
| **1.&nbsp;Rules** | Stable and policy-driven — you can write the logic down | Business rules / thresholds | An audit trail |
| **2.&nbsp;Predict** | A signal you can't hand-code, from structured/tabular ERP data | **SAP-RPT-1** on SAP AI Core | Model accuracy and drift |
| **3.&nbsp;Reason** | Adaptive and multi-criteria — the next step depends on what you learn | **Agent** on Gen AI Hub | A reasoning trace + human approval |

**The rule of the ladder:** *Climb only as high as the decision requires — and the higher you climb, the more you govern.* A rules engine needs an audit log; an agent needs a human in the loop.

You will build all three rungs in this workshop:
- **Rules** — the Green/Amber/Red risk policy (Exercise 1A)
- **Predict** — SAP-RPT-1 delay scoring (Exercise 1A & 1B)
- **Reason** — the ReAct mitigation agent (Exercise 2)


## Business Context: Why This Matters

### Why BestRun Can't Just Hold More Inventory

The obvious fix for supply risk is a buffer. For BestRun, that lever is off the table-which is what makes prediction the primary defense rather than one option among many:

| Constraint | Effect |
|------------|--------|
| **Sole-sourced critical components** (secure element, MEMS sensor) | Can't buffer against a supplier that already rations allocation; 30+ week lead times |
| **Short product-refresh & obsolescence cycle** | Safety stock risks becoming a write-off before it's consumed |
| **Fabless / contract manufacturing** | Line time is pre-booked and costly-a missing component forfeits the slot |
| **Hypergrowth outpacing supply maturity** | Demand grows faster than second sources can be qualified |

Because buffer stock isn't a viable hedge, **early warning is the only lever left**-and that is exactly what a delay-prediction model provides.

### The Cost of Being Reactive

| Scenario | Impact |
|----------|--------|
| Critical sensor delayed 2 days | $240,000 lost production |
| Control chip arrives 1 week late | Missed customer SLA, $50,000 penalty |
| Proactive supplier switch | $5,000 premium, but $0 downtime |

### The Data Landscape

BestRun's S/4HANA system contains:
- **600+ historical POs** with actual delivery outcomes
- **3 key suppliers** with varying reliability profiles
- **6 material types** ranging from critical sensors to commodity connectors
- **Seasonal patterns** (Q4 logistics congestion, monsoon impacts in Asia)

### SAP BTP Services Used

| Service | Role | Why SAP-Native? |
|---------|------|----------------|
| **SAP AI Core** | Model hosting and inference runtime | Enterprise auth, resource isolation, audit trail |
| **SAP-RPT-1** | Tabular prediction model | Purpose-built for structured business data |
| **Gen AI Hub SDK** | Pro-code orchestration | Consistent API, version-pinned, SAP support |

### The Deterministic Pipeline at a Glance

This exercise builds a fixed, policy-driven flow: predict the delay with SAP-RPT-1, classify it into a risk tier, and — when risk is high — generate a mitigation proposal from alternative suppliers.

```mermaid
flowchart LR
    PO["New Purchase Order"] --> RPT["SAP-RPT-1<br/>Predict delay (days)"]
    RPT --> TIER{"Risk Tier"}
    TIER -->|"&lt; 1 day"| G["Green<br/>Standard monitoring"]
    TIER -->|"1-3 days"| A["Amber<br/>Increased attention"]
    TIER -->|"&gt; 3 days"| R["Red<br/>Mitigation required"]
    R --> MIT["Alternative-supplier<br/>mitigation proposal"]
    style RPT fill:#EAF3FB,stroke:#0A6ED1,stroke-width:2px
    style G fill:#E8F5E9,stroke:#30914C
    style A fill:#FFF3E0,stroke:#E76500
    style R fill:#FDECEA,stroke:#BB0000
```


### Applying the Ladder to This Scenario

Exercise 1A lives on the **Rules + Predict** rungs. Use this checkpoint to confirm that is the right altitude before climbing to an agent:

| Question | If yes | If no |
|---------|--------|-------|
| Is the data primarily structured, tabular business data from ERP? | **Predict** fits — `sap-rpt-1` is a strong choice | Consider LLM or unstructured retrieval patterns |
| Is the target output a numeric risk signal or forecast? | Stay on the **Predict** rung | Consider descriptive or generative patterns |
| Is the process mostly known and policy-driven? | **Rules + Predict** is enough (this exercise) | Climb to **Reason** (Exercise 2) |
| Do sourcing changes need human sign-off? | **Yes** — keep human-in-the-loop; agent stays advisory | Full autonomy may be safe — but justify why |

**Architect guidance:** start at the lowest rung that proves business value, then climb only where a higher rung measurably improves the decision.


---

## Section 1 — Setup & Configuration

📘 **KNOWLEDGE POINT — Credentials are configuration, not code**
Enterprise AI code never hardcodes secrets. We load credentials from a git-ignored `.env` file and **fail fast**: if anything required is missing, the notebook stops with a clear message instead of failing deep inside an API call later.

🎯 **Outcome:** a validated, live connection to SAP AI Core — every later section depends on it.

⌨️ **Your Turn:** run Steps 1.1 → 1.3 in order. Restart the kernel if prompted after the install.

### Step 1.1 — Install required packages

- `generative-ai-hub-sdk`: SAP-supported SDK for AI Core and Gen AI Hub
- `pandas`: tabular PO data
- `python-dotenv`: credential management from `.env`

In [ ]:
%pip install generative-ai-hub-sdk==4.12.4 --quiet
%pip install pandas python-dotenv requests --quiet

print("Packages installed successfully.")

### Step 1.2 — Prerequisites check

A fail-fast guard: Python version, importable packages, and data files present — *before* we touch the network.

In [ ]:
import sys
import socket

def check_prerequisites():
    """Validate environment before workshop starts."""
    checks_passed = 0
    checks_total = 3
    
    # Check 1: Python version
    py_version = sys.version_info
    if py_version >= (3, 9):
        print(f"[PASS] Python version: {py_version.major}.{py_version.minor}.{py_version.micro}")
        checks_passed += 1
    else:
        print(f"[FAIL] Python version {py_version.major}.{py_version.minor} - requires 3.9+")
    
    # Check 2: Required packages importable
    try:
        import pandas as pd
        import requests
        from dotenv import load_dotenv
        print("[PASS] Required packages are importable")
        checks_passed += 1
    except ImportError as e:
        print(f"[FAIL] Missing package: {e.name}. Re-run the install cell above.")
    
    # Check 3: Data files exist
    import os
    data_files = ["data/historical_po_data.csv", "data/new_po_prediction.csv", "data/alt_supplier_table.csv"]
    missing = [f for f in data_files if not os.path.exists(f)]
    if not missing:
        print("[PASS] All data files found")
        checks_passed += 1
    else:
        print(f"[FAIL] Missing data files: {missing}")
    
    print("-" * 50)
    if checks_passed == checks_total:
        print(f"All {checks_total} checks passed. Ready to proceed!")
    else:
        print(f"{checks_passed}/{checks_total} checks passed. Review issues above.")

check_prerequisites()

### Step 1.3 — Imports and global constants

All libraries plus the constants used throughout: the model name, HTTP timeouts, the **risk-tier thresholds** (business policy), and the line-down cost parameters.

In [ ]:
import json
import os
import time
from typing import Any, Dict, List, Tuple

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import display, HTML

# --- Global Constants ---
MODEL_NAME = "sap-rpt-1-small"
TOKEN_TIMEOUT_SECONDS = 30
HTTP_TIMEOUT_SECONDS = 60
TOKEN_REFRESH_BUFFER_SECONDS = 60

# --- Risk Tier Thresholds (Business Policy) ---
RISK_THRESHOLD_AMBER = 1.0  # Days
RISK_THRESHOLD_RED = 3.0   # Days

# --- Cost Parameters (for business case calculation) ---
LINE_DOWN_COST_PER_HOUR = 15000  # USD
TYPICAL_DISRUPTION_HOURS = 8

print("Imports and constants loaded.")

### Step 1.4 — Load `.env` configuration

**Expected variables:** `AICORE_AUTH_URL`, `AICORE_CLIENT_ID`, `AICORE_CLIENT_SECRET`, `AICORE_BASE_URL`, `AICORE_RESOURCE_GROUP`, `RPT1_DEPLOYMENT_URL`.

> **Setup:** copy `.env.example` to `.env`, fill in every value, save, and re-run this cell.
> **Endpoint note:** set `RPT1_DEPLOYMENT_URL` to the deployment **base** URL; the notebook appends `/predict` automatically.

In [ ]:
dotenv_path = find_dotenv()
load_dotenv(dotenv_path=dotenv_path, override=True)

AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
RPT1_DEPLOYMENT_URL = os.getenv("RPT1_DEPLOYMENT_URL")

required = {
    "AICORE_AUTH_URL": AICORE_AUTH_URL,
    "AICORE_CLIENT_ID": AICORE_CLIENT_ID,
    "AICORE_CLIENT_SECRET": AICORE_CLIENT_SECRET,
    "AICORE_BASE_URL": AICORE_BASE_URL,
    "AICORE_RESOURCE_GROUP": AICORE_RESOURCE_GROUP,
    "RPT1_DEPLOYMENT_URL": RPT1_DEPLOYMENT_URL,
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise EnvironmentError(
        "Missing required environment variables: "
        f"{', '.join(missing)}.\n"
        "This exercise connects to the live SAP-RPT-1 model on SAP AI Core and "
        "cannot run without valid credentials.\n"
        "Fix: create a .env file in the project root (copy .env.example), fill in "
        "every key listed above, then restart the kernel and re-run this cell."
    )

print(f"Loaded .env from: {dotenv_path}")
print(f"Resource group: {AICORE_RESOURCE_GROUP}")
print(f"RPT-1 Deployment: ...{RPT1_DEPLOYMENT_URL[-20:]}")
print("\nConfiguration valid. Ready to connect to live SAP AI Core.")


> ### ✓ CHECKPOINT — Section 1
> You have a validated, live connection to SAP AI Core. If the cell above printed *"Configuration valid,"* you're ready.
>
> 🏭 **What would break in production?** A notebook reads `.env` once. A deployed service needs secrets from a **credential store** (BTP destination service / Credential Store), rotation without redeploy, and least-privilege scoping. Hold that thought for the Architecture Playbook.
>
> *Troubleshooting: `401` → check client ID/secret · `403` → check resource group · `404` → check the deployment is running in AI Launchpad.*

---

## Section 2 — The Data & the Dark-Data Problem

📘 **KNOWLEDGE POINT — "Dark data" is signal you already own but don't use**
BestRun's S/4HANA holds years of PO outcomes: which vendor, which material, how much, what month, and *how late it actually arrived*. That history is a **labelled training signal** sitting idle. Before any model, an architect learns to *read* that signal — because a model can only learn patterns that are actually present in the data.

🎯 **Outcome:** you can explain, from the data alone, which orders are risky and why — so you know what the model should be able to learn.

⌨️ **Your Turn:** run Steps 2.1 → 2.4 and, at 2.4, **predict by eye** before the model ever runs.

### Step 2.1 — Load the datasets

1. **Historical PO data** — past orders with known delivery outcomes (the labelled signal)
2. **New PO** — the order we need to assess
3. **Alternative supplier table** — backup suppliers per material

In [ ]:
# Load datasets
historical_df = pd.read_csv("data/historical_po_data.csv")
prediction_df = pd.read_csv("data/new_po_prediction.csv")
alt_supplier_df = pd.read_csv("data/alt_supplier_table.csv")

print(f"Historical POs: {len(historical_df):,} records")
print(f"POs to predict: {len(prediction_df):,} records")
print(f"Alternative suppliers: {len(alt_supplier_df):,} records")
print("\n" + "="*60)
print("Sample of historical data:")
historical_df.head()

### Step 2.2 - Understand the Feature Set

The SAP-RPT-1 model uses these features to predict delay risk:

| Feature | Source (S/4HANA) | Description |
|---------|------------------|-------------|
| `Vendor_ID` | LFA1 | Supplier identifier |
| `Vendor_Country` | LFA1 | Supplier location (logistics risk) |
| `Vendor_OTIF_Percent` | Derived | On-Time-In-Full historical rate |
| `Vendor_Avg_Past_Delay` | Derived | Average historical delay (days) |
| `Material_ID` | MARA | Material number |
| `Material_Group` | MARA | Material category |
| `Criticality_Flag` | Custom | Is this a line-stopping component? |
| `Order_Quantity` | EKPO | Units ordered |
| `Net_Price` | EKPO | Unit price (USD) |
| `Planned_Lead_Time_Days` | EKPO/INFO | Expected delivery time |
| `Order_Month` | EKKO | Seasonality indicator |
| `Incoterms` | EKKO | Delivery responsibility terms |

In [ ]:
print("Dataset Schema:")
print("-" * 40)
for col in historical_df.columns:
    dtype = historical_df[col].dtype
    sample = historical_df[col].iloc[0]
    print(f"{col:30} | {str(dtype):10} | Example: {sample}")

### Step 2.3 — Historical risk distribution

The **risk-tier policy** (this is Rung 1 logic, applied here to *known outcomes* just to see the shape of history):
- **Green** (< 1 day) · **Amber** (1–3 days) · **Red** (> 3 days)

In [ ]:
def derive_risk_tier(delay_days: float) -> str:
    """Classify delay into business risk tiers."""
    if delay_days < RISK_THRESHOLD_AMBER:
        return "Green"
    elif delay_days <= RISK_THRESHOLD_RED:
        return "Amber"
    return "Red"

# Apply to historical data
historical_df["Risk_Tier"] = historical_df["Actual_Delay_Days"].apply(derive_risk_tier)

# Display distribution
risk_counts = historical_df["Risk_Tier"].value_counts()
risk_pcts = (risk_counts / len(historical_df) * 100).round(1)

print("Historical Risk Distribution:")
print("=" * 40)
for tier in ["Green", "Amber", "Red"]:
    count = risk_counts.get(tier, 0)
    pct = risk_pcts.get(tier, 0)
    bar = "|" * int(pct / 2)
    print(f"{tier:8} | {count:4} POs ({pct:5.1f}%) {bar}")

print("\n" + "=" * 40)
print(f"Average delay: {historical_df['Actual_Delay_Days'].mean():.2f} days")
print(f"Max delay: {historical_df['Actual_Delay_Days'].max():.1f} days")

### Step 2.4 — Vendor performance, and a prediction by eye

Run the cell to see each vendor's reliability profile. Then, **before any model runs**, make a call:

> ⌨️ **Predict by eye:** Our target order is **VENDOR_C, Control Chip (critical), 2,200 units, ordered in November.** Just from this table — Green, Amber, or Red? Note your answer. We'll test it against a rule (Section 3) and the model (Section 4).

In [ ]:
vendor_stats = historical_df.groupby("Vendor_ID").agg({
    "Actual_Delay_Days": ["mean", "std", "max"],
    "PO_ID": "count",
    "Vendor_Country": "first",
    "Vendor_OTIF_Percent": "first"
}).round(2)

vendor_stats.columns = ["Avg_Delay", "Std_Delay", "Max_Delay", "PO_Count", "Country", "OTIF_%"]
vendor_stats = vendor_stats.sort_values("Avg_Delay")

print("Vendor Performance Summary:")
print("=" * 70)
display(vendor_stats)

print("\nKey Insight:")
best = vendor_stats.index[0]
worst = vendor_stats.index[-1]
print(f"  - {best} (USA) has the best performance: {vendor_stats.loc[best, 'Avg_Delay']:.1f} day avg delay")
print(f"  - {worst} (Vietnam) has highest risk: {vendor_stats.loc[worst, 'Avg_Delay']:.1f} day avg delay")

> ### ✓ CHECKPOINT — Section 2
> You can read the signal directly: VENDOR_A is uniformly reliable; VENDOR_B and VENDOR_C are slower on average. But averages hide *when* and *how much* — keep that in mind.
>
> 🏭 **What would break in production?** This data was clean and local. Real S/4HANA extraction means schema drift, missing values, late-arriving actuals, and PII/authorization boundaries. **Data readiness is the real project.**

---

## Section 3 — Rung 1: Rules Only (no AI) 🔵

📘 **KNOWLEDGE POINT — Always start at the bottom of the ladder**
The cheapest, most auditable automation is a **deterministic rule** a human can write down and defend. Before reaching for a model, an architect asks: *can a rule do this well enough?* Here we build the rule an analyst would write today — classify risk from the vendor's headline reliability (OTIF %) — and then we find exactly where it goes blind.

🎯 **Outcome:** you'll experience the rung-1 baseline *and* discover its structural limit — which is what earns the climb to prediction.

⌨️ **Your Turn:** configure the PO once (Step 3.1), judge it with a rule (3.2), then expose the blind spot (3.3).

### Step 3.1 — Configure the purchase order

This sets up the single PO that **both rungs** will judge — first the rule (this section), then the live model (Section 4). Change `SELECTED_VENDOR` / `SELECTED_QUANTITY` to explore, then re-run this cell.

| Vendor | Quantity | What to expect |
|--------|----------|----------------|
| VENDOR_A | 1000 | Low risk |
| VENDOR_B | 2200 | **Rule says Amber — but watch Section 4** |
| VENDOR_C | 2200 | High risk (default) |

In [ ]:
# ============================================================
# WORKSHOP PARAMETERS - Modify these to explore scenarios
# ============================================================

SELECTED_VENDOR = "VENDOR_C"   # Options: VENDOR_A, VENDOR_B, VENDOR_C
SELECTED_QUANTITY = 2200       # Options: 500, 1000, 1500, 2200, 2600

# ============================================================

# Update the prediction row
prediction_df.loc[0, "Vendor_ID"] = SELECTED_VENDOR
prediction_df.loc[0, "Order_Quantity"] = SELECTED_QUANTITY

# Sync vendor profile attributes
vendor_profile = historical_df.groupby("Vendor_ID")[
    ["Vendor_OTIF_Percent", "Vendor_Avg_Past_Delay", "Vendor_Country"]
].first().reset_index()

vendor_attrs = vendor_profile[vendor_profile["Vendor_ID"] == SELECTED_VENDOR].iloc[0]
prediction_df.loc[0, "Vendor_OTIF_Percent"] = vendor_attrs["Vendor_OTIF_Percent"]
prediction_df.loc[0, "Vendor_Avg_Past_Delay"] = vendor_attrs["Vendor_Avg_Past_Delay"]
prediction_df.loc[0, "Vendor_Country"] = vendor_attrs["Vendor_Country"]

print("Prediction Scenario Configured:")
print("=" * 50)
print(f"PO Number:      {prediction_df.loc[0, 'PO_ID']}")
print(f"Vendor:         {SELECTED_VENDOR} ({vendor_attrs['Vendor_Country']})")
print(f"Vendor OTIF:    {vendor_attrs['Vendor_OTIF_Percent']}%")
print(f"Material:       {prediction_df.loc[0, 'Material_ID']} ({prediction_df.loc[0, 'Material_Group']})")
print(f"Critical:       {prediction_df.loc[0, 'Criticality_Flag']}")
print(f"Quantity:       {SELECTED_QUANTITY:,} units")
print(f"Order Month:    {prediction_df.loc[0, 'Order_Month']} (November)")
print(f"Incoterms:      {prediction_df.loc[0, 'Incoterms']}")

### Step 3.2 — Judge the PO with a rule (no model)

A rule keyed on vendor OTIF bands. Fully deterministic, fully auditable — and it looks at exactly **one** number.

In [ ]:
# RUNG 1: RULES ONLY (no AI) -----------------------------------------
# A deterministic rule a supply chain analyst can write and defend TODAY,
# using only the vendor's headline reliability (OTIF %). No model involved.

def rules_only_risk_tier(po) -> str:
    """Classify delay risk from vendor OTIF bands. Deterministic and auditable."""
    otif = po["Vendor_OTIF_Percent"]
    if otif >= 95:
        return "Green"   # highly reliable vendor
    elif otif >= 85:
        return "Amber"   # watch
    else:
        return "Red"     # unreliable vendor

rules_tier = rules_only_risk_tier(prediction_df.iloc[0])

print("RUNG 1 - RULES ONLY (no AI)")
print("=" * 46)
print(f"PO:            {prediction_df.loc[0, 'PO_ID']}")
print(f"Vendor:        {prediction_df.loc[0, 'Vendor_ID']}  (OTIF {prediction_df.loc[0, 'Vendor_OTIF_Percent']}%)")
print(f"Quantity:      {int(prediction_df.loc[0, 'Order_Quantity']):,} units, ordered month {int(prediction_df.loc[0, 'Order_Month'])}")
print(f"RULE VERDICT:  {rules_tier}")
print()
print("Note what the rule used: ONE number (vendor OTIF).")
print("It ignored the material, the quantity, and the month entirely.")


### Step 3.3 — Where the rule goes blind

A vendor-OTIF rule gives **every** order from a vendor the **same** tier. But risk also lives *inside* a vendor — in the combination of *when* and *how much* you order. Let's prove it from history.

In [ ]:
# The blind spot: risk that lives INSIDE a vendor, invisible to a vendor-level rule.
# VENDOR_B has one OTIF (88%), so the rule gives every VENDOR_B order the SAME tier.
# But history shows its delay explodes for large Q4 orders.

vb = historical_df[historical_df["Vendor_ID"] == "VENDOR_B"].copy()
qty_hi = vb["Order_Quantity"].quantile(0.66)
vb["Segment"] = [
    "Q4 + high qty" if (m in (10, 11, 12) and q >= qty_hi) else "rest of year"
    for m, q in zip(vb["Order_Month"], vb["Order_Quantity"])
]

summary = (
    vb.groupby("Segment")["Actual_Delay_Days"]
      .agg(Avg_Delay="mean", Max_Delay="max", POs="count")
      .round(2)
)

rule_tier_b = rules_only_risk_tier(pd.Series({"Vendor_OTIF_Percent": 88}))

print("VENDOR_B - one vendor, one OTIF (88%), one rule verdict:", rule_tier_b)
print("=" * 55)
print(summary.to_string())
print()
print(f"The rule says '{rule_tier_b}' for EVERY VENDOR_B order.")
print("History says Q4 + high-quantity orders run in the RED zone (>3 days).")
print()
print("The rule cannot see this: the risk is in the month x quantity")
print("combination, not the vendor. THAT is the signal a model can learn.")


> ### ✓ CHECKPOINT — Section 3
> You built rung 1 and found its ceiling. Rules are **cheap, instant, and auditable** — keep them for stable, policy-driven logic. But a rule keyed on a vendor is **structurally blind to interaction effects** (month × quantity × material). That blindness is precisely what a tabular model learns from history.
>
> 🏭 **What would break in production?** Nothing technical — rules are robust. The failure mode is *human*: teams pile exception on exception until the rulebook is unmaintainable and no one can say why an order was flagged. When you can't hand-code the rule, **climb one rung.**

---

## Section 4 — Rung 2: Predict with SAP-RPT-1 🟢

📘 **KNOWLEDGE POINT — SAP-RPT-1 reads the *whole* order, not one column**
SAP-RPT-1 is a Relational Pretrained Transformer for **tabular** data. You hand it a block of **historical rows with known outcomes** plus your **new row with the target masked** (`[PREDICT]`), and it infers the target *in context* — learning the month × quantity × vendor × material interactions the rule couldn't. No training job, no feature engineering: prediction happens **in the payload**.

🎯 **Outcome:** you'll call the live model and see it judge the same PO the rule just judged.

⌨️ **Your Turn:** run 4.1 → 4.3, then read the rung-1-vs-rung-2 comparison at 4.4.

### Step 4.1 — SAP AI Core client

Handles OAuth token management (with refresh) and the inference POST. This is the reusable enterprise-auth boundary.

In [ ]:
class AICoreClient:
    """
    Client for SAP AI Core API.
    
    Handles OAuth token management and inference requests.
    """
    
    def __init__(self, auth_url: str, client_id: str, client_secret: str,
                 base_url: str, resource_group: str):
        self._auth_url = auth_url
        self._client_id = client_id
        self._client_secret = client_secret
        self._base_url = base_url
        self._resource_group = resource_group
        self._access_token: str | None = None
        self._token_expires_at: float = 0.0
    
    def _get_token(self) -> str:
        """Get a valid access token, refreshing if necessary."""
        now = time.time()
        if self._access_token and now < (self._token_expires_at - TOKEN_REFRESH_BUFFER_SECONDS):
            return self._access_token
        
        response = requests.post(
            f"{self._auth_url}/oauth/token",
            data={"grant_type": "client_credentials"},
            auth=(self._client_id, self._client_secret),
            timeout=TOKEN_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        payload = response.json()
        
        self._access_token = payload["access_token"]
        self._token_expires_at = now + int(payload.get("expires_in", 600))
        return self._access_token
    
    def predict(self, deployment_url: str, payload: dict) -> dict:
        """
        Run inference on SAP AI Core deployment.
        
        Args:
            deployment_url: Full URL to the model deployment endpoint
            payload: Request payload for the model
            
        Returns:
            Model response as dictionary
        """
        token = self._get_token()
        
        response = requests.post(
            deployment_url,
            json=payload,
            headers={
                "Authorization": f"Bearer {token}",
                "AI-Resource-Group": self._resource_group,
                "Content-Type": "application/json"
            },
            timeout=HTTP_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        return response.json()


# Initialize the SAP AI Core client
aicore_client = AICoreClient(
    auth_url=AICORE_AUTH_URL,
    client_id=AICORE_CLIENT_ID,
    client_secret=AICORE_CLIENT_SECRET,
    base_url=AICORE_BASE_URL,
    resource_group=AICORE_RESOURCE_GROUP,
)
print("AI Core client initialized.")


### Step 4.2 — The prediction function

Note the **context + `[PREDICT]` placeholder** pattern in the payload — this *is* how SAP-RPT-1 works. We send a sample of history plus the masked new row and read back the predicted delay.

In [ ]:
def predict_delay(row: pd.Series, context_df: pd.DataFrame) -> Tuple[float, str]:
    """
    Predict delivery delay for a purchase order using the live SAP-RPT-1 model.

    Args:
        row: The new PO to predict
        context_df: Historical POs for context

    Returns:
        Tuple of (predicted_delay_days, prediction_method)
    """
    # Prepare payload for SAP-RPT-1
    columns = [
        "Vendor_ID", "Vendor_Country", "Vendor_OTIF_Percent", "Vendor_Avg_Past_Delay",
        "Material_ID", "Material_Group", "Criticality_Flag", "Plant_ID",
        "Order_Quantity", "Net_Price", "Planned_Lead_Time_Days", "Order_Month",
        "Incoterms", "Actual_Delay_Days"
    ]

    # Sample context for efficiency
    context_sample = context_df[columns].sample(n=min(200, len(context_df)), random_state=42)

    # Prepare prediction row
    pred_row = row[columns[:-1]].to_dict()
    pred_row["Actual_Delay_Days"] = "[PREDICT]"

    # SAP-RPT-1 expects rows + prediction_config payload
    rows = pd.concat([context_sample, pd.DataFrame([pred_row])], ignore_index=True).to_dict("records")
    payload = {
        "rows": rows,
        "prediction_config": {
            "target_columns": [
                {
                    "name": "Actual_Delay_Days",
                    "prediction_placeholder": "[PREDICT]"
                }
            ]
        }
    }

    # Deployment URL in .env is base path; model endpoint is /predict
    deployment_url = RPT1_DEPLOYMENT_URL.rstrip("/")
    if not deployment_url.endswith("/predict"):
        deployment_url = f"{deployment_url}/predict"

    try:
        # Call SAP-RPT-1
        response = aicore_client.predict(
            deployment_url=deployment_url,
            payload=payload
        )
    except requests.HTTPError as e:
        error_detail = ""
        if e.response is not None:
            error_detail = f"\n  status={e.response.status_code}\n  body={e.response.text[:300]}"
        raise RuntimeError(
            f"SAP-RPT-1 inference call failed.{error_detail}\n"
            "Check that RPT1_DEPLOYMENT_URL points to an active deployment and that "
            "your AI Core credentials and resource group are correct."
        ) from e

    prediction_value = (
        response["predictions"][0]["Actual_Delay_Days"][0]["prediction"]
    )
    predicted_delay = float(prediction_value)
    return predicted_delay, "sap-rpt-1"


print("Prediction function ready (live SAP-RPT-1).")


### Step 4.3 — Run the live prediction

Calls SAP-RPT-1 on the PO you configured in Step 3.1 and maps the predicted delay to a risk tier.

In [ ]:
# Run prediction
predicted_delay, method = predict_delay(prediction_df.iloc[0], historical_df)
risk_tier = derive_risk_tier(predicted_delay)

# Color coding for display
tier_colors = {"Green": "#2e7d32", "Amber": "#f57f17", "Red": "#c62828"}
tier_color = tier_colors.get(risk_tier, "#757575")

print("\n" + "=" * 50)
print("PREDICTION RESULT")
print("=" * 50)
print(f"Predicted Delay:  {predicted_delay} days")
print(f"Risk Tier:        {risk_tier}")
print(f"Prediction Method: {method}")
print(f"Material Critical: {prediction_df.loc[0, 'Criticality_Flag']}")

# Visual display
html_result = f"""
<div style="border:2px solid {tier_color}; border-radius:8px; padding:16px; margin:10px 0; background:#f5f5f5;">
    <h3 style="margin-top:0;">Prediction Result: PO {prediction_df.loc[0, 'PO_ID']}</h3>
    <div style="display:flex; gap:40px;">
        <div>
            <strong>Predicted Delay:</strong><br>
            <span style="font-size:2em; font-weight:bold;">{predicted_delay} days</span>
        </div>
        <div>
            <strong>Risk Tier:</strong><br>
            <span style="font-size:2em; font-weight:bold; color:{tier_color};">{risk_tier}</span>
        </div>
        <div>
            <strong>Material:</strong><br>
            <span>{prediction_df.loc[0, 'Material_Group']}</span><br>
            <span style="color:{'red' if prediction_df.loc[0, 'Criticality_Flag']=='Yes' else 'gray'};">
                {'CRITICAL' if prediction_df.loc[0, 'Criticality_Flag']=='Yes' else 'Standard'}
            </span>
        </div>
    </div>
</div>
"""
display(HTML(html_result))

### Step 4.4 — Rung 1 vs Rung 2, same PO

The whole point of the ladder, side by side.

In [ ]:
# Compare the two rungs on the SAME purchase order.
print("SAME PO - TWO RUNGS OF THE DECISION LADDER")
print("=" * 46)
print(f"Rung 1  Rules only:   {rules_tier}")
print(f"Rung 2  SAP-RPT-1:    {risk_tier}   ({predicted_delay} days predicted)")
print("-" * 46)
if rules_tier == risk_tier:
    print("They AGREE here - but the rule agreed using only vendor OTIF.")
    print("Test the blind spot yourself: go to Step 3.1, set")
    print("SELECTED_VENDOR='VENDOR_B', SELECTED_QUANTITY=2200, re-run")
    print("Steps 3.1 -> 4.4, and watch the rule say 'Amber' while the")
    print("model reads the Q4 + high-quantity risk the rule cannot see.")
else:
    print("They DISAGREE. The model read the whole order - vendor, material,")
    print("quantity, and month together - and caught risk the vendor rule missed.")


> ### ✓ CHECKPOINT — Section 4
> You climbed to rung 2. SAP-RPT-1 predicts a delay from the **full feature set in context**, catching interaction effects a rule can't encode. Prediction is now a *signal* — but a number on its own changes nothing.
>
> 🏭 **What would break in production?** A model you don't monitor is a liability. You need **accuracy tracking, drift detection**, and a retraining trigger as vendors and seasons shift. *(Exercise 1B measures that accuracy against open-source baselines.)*

---

## Section 5 — Rules + Predict = Policy 🔵+🟢

📘 **KNOWLEDGE POINT — A prediction only matters once it drives a governed action**
Rungs combine. The model gives a *number*; a **rule** turns that number — plus business context like criticality — into an **action level** and a **dollar impact**. This is where prediction becomes a decision the business can act on and audit.

🎯 **Outcome:** you can translate a risk score into a governed action and quantify the value of acting.

⌨️ **Your Turn:** run the policy cell. Then re-run with different scenarios from Step 3.1 and watch the action level change.

### Step 4.1 - Apply Business Policy

BestRun's supply chain policy:

| Condition | Action |
|-----------|--------|
| Green risk + non-critical | Standard monitoring |
| Amber risk OR critical material | Increased attention, notify planner |
| Red risk + critical material | **Immediate mitigation required** |

In [ ]:
def apply_business_policy(risk_tier: str, is_critical: bool) -> Tuple[str, str]:
    """
    Apply BestRun's supply chain risk policy.
    
    Returns:
        Tuple of (action_level, recommendation)
    """
    if risk_tier == "Red" and is_critical:
        return "CRITICAL", "Immediate mitigation required. Initiate alternative sourcing."
    elif risk_tier == "Red":
        return "HIGH", "High risk detected. Review alternative suppliers."
    elif risk_tier == "Amber" or is_critical:
        return "ELEVATED", "Increased monitoring. Prepare contingency plan."
    else:
        return "NORMAL", "Standard supplier monitoring. No action required."


def estimate_mitigation_value(delay_days: float, is_critical: bool) -> int:
    """
    Estimate the business value of proactive mitigation.
    
    Assumes:
    - Line-down cost of $15,000/hour
    - Typical disruption event of 8 hours
    - Critical materials cause full line stoppage
    """
    if not is_critical or delay_days < RISK_THRESHOLD_RED:
        return 0
    
    # Estimate hours of disruption based on delay severity
    disruption_hours = min(delay_days * 4, TYPICAL_DISRUPTION_HOURS * 2)  # Cap at 2x typical
    return int(disruption_hours * LINE_DOWN_COST_PER_HOUR)


# Apply policy
is_critical = prediction_df.loc[0, "Criticality_Flag"] == "Yes"
action_level, recommendation = apply_business_policy(risk_tier, is_critical)
avoided_loss = estimate_mitigation_value(predicted_delay, is_critical)

print("Business Policy Assessment:")
print("=" * 50)
print(f"Action Level:     {action_level}")
print(f"Recommendation:   {recommendation}")
if avoided_loss > 0:
    print(f"\nPotential Avoided Loss: ${avoided_loss:,.0f}")
    print(f"  (Based on {predicted_delay:.1f} days delay x $15,000/hour line-down cost)")

> ### ✓ CHECKPOINT — Section 5
> Prediction + rule = a governed action with a business case attached. Notice the policy escalates on **criticality**, not just the tier — a Green non-critical part and a Green *line-stopping* part are not the same decision.
>
> 🏭 **What would break in production?** These thresholds (`1.0`, `3.0` days, `$15,000/hr`) are **governed parameters**, not code constants — they need an owner, a change log, and review as the business changes.

---

## Section 6 — Mitigation Proposal 🔵+🟢

📘 **KNOWLEDGE POINT — The system proposes; the human decides**
When risk is high, we *propose* an alternative supplier with a full cost/benefit comparison — but we never *act*. Sourcing changes carry price, quality, and relationship consequences, so the proposal is **advisory** and a human approves. This human-in-the-loop stance is the governance backbone that carries straight into the Exercise 2 agent.

🎯 **Outcome:** you can generate a decision-ready proposal that keeps accountability with a person.

⌨️ **Your Turn:** run the proposal and the dashboard, then read the decision-rights table.

### Decision Rights and Business Trade-Offs

Prediction is advisory. Decision accountability stays with planners and approvers.

| Decision Option | Typical Benefit | Typical Cost/Risk | Default Owner |
|----------------|-----------------|-------------------|---------------|
| Stay with current supplier | No switching effort | Higher delay risk | Planner |
| Expedite current supplier | Faster recovery | Premium freight cost | Planner + Procurement |
| Switch to alternate supplier | Lower delay probability | Price/quality onboarding risk | Procurement Approver |
| Split order across suppliers | Risk diversification | Coordination complexity | Supply Chain Lead |

**Governance rule in this workshop:** model predicts, agent proposes, human approves any sourcing-impacting action.

In [ ]:
def find_alternative_suppliers(material_id: str, current_vendor: str, 
                                alt_df: pd.DataFrame) -> pd.DataFrame:
    """
    Find alternative suppliers for a material.
    
    Filters:
    - Same material
    - Different vendor (not current)
    
    Sorts by:
    - Lead time (ascending)
    - Price (ascending)
    - Stock (descending)
    """
    candidates = alt_df[
        (alt_df["Material_ID"] == material_id) &
        (alt_df["Alt_Vendor"] != current_vendor)
    ].copy()
    
    return candidates.sort_values(
        ["Lead_Time_Days", "Indicative_Unit_Price"],
        ascending=[True, True]
    )


def generate_mitigation_proposal(prediction_row: pd.Series, predicted_delay: float,
                                  risk_tier: str, alt_df: pd.DataFrame) -> Dict[str, Any]:
    """
    Generate a structured mitigation proposal.
    """
    material_id = prediction_row["Material_ID"]
    current_vendor = prediction_row["Vendor_ID"]
    quantity_needed = int(prediction_row["Order_Quantity"])
    is_critical = prediction_row["Criticality_Flag"] == "Yes"
    
    # Find alternatives
    alternatives = find_alternative_suppliers(material_id, current_vendor, alt_df)
    
    if len(alternatives) == 0:
        return {
            "status": "NO_ALTERNATIVES",
            "message": f"No alternative suppliers found for {material_id}"
        }
    
    # Select best alternative
    best = alternatives.iloc[0]
    
    # Calculate cost comparison
    current_unit_price = prediction_row["Net_Price"]
    alt_unit_price = best["Indicative_Unit_Price"]
    price_premium = (alt_unit_price - current_unit_price) * quantity_needed
    
    # Stock availability check
    stock_available = int(best["Current_Available_Stock"])
    can_fulfill = stock_available >= quantity_needed
    
    return {
        "status": "PROPOSAL_READY",
        "risk_summary": {
            "predicted_delay": predicted_delay,
            "risk_tier": risk_tier,
            "is_critical": is_critical,
            "avoided_loss": estimate_mitigation_value(predicted_delay, is_critical)
        },
        "current_order": {
            "vendor": current_vendor,
            "material": material_id,
            "quantity": quantity_needed,
            "unit_price": current_unit_price
        },
        "recommended_alternative": {
            "vendor": best["Alt_Vendor"],
            "country": best["Alt_Vendor_Country"],
            "lead_time_days": int(best["Lead_Time_Days"]),
            "unit_price": alt_unit_price,
            "stock_available": stock_available,
            "can_fulfill": can_fulfill
        },
        "financial_impact": {
            "price_premium": price_premium,
            "avoided_loss": estimate_mitigation_value(predicted_delay, is_critical),
            "net_benefit": estimate_mitigation_value(predicted_delay, is_critical) - max(price_premium, 0)
        },
        "all_alternatives": alternatives.to_dict("records")
    }


# Generate proposal if high risk
if action_level in ["CRITICAL", "HIGH"]:
    proposal = generate_mitigation_proposal(
        prediction_df.iloc[0], predicted_delay, risk_tier, alt_supplier_df
    )
    
    print("\n" + "=" * 60)
    print("MITIGATION PROPOSAL")
    print("=" * 60)
    
    if proposal["status"] == "PROPOSAL_READY":
        rec = proposal["recommended_alternative"]
        curr = proposal["current_order"]
        fin = proposal["financial_impact"]
        
        print(f"\nCurrent Order:")
        print(f"  Vendor: {curr['vendor']}")
        print(f"  Material: {curr['material']}")
        print(f"  Quantity: {curr['quantity']:,} units")
        print(f"  Unit Price: ${curr['unit_price']:.2f}")
        
        print(f"\nRecommended Alternative:")
        print(f"  Vendor: {rec['vendor']} ({rec['country']})")
        print(f"  Lead Time: {rec['lead_time_days']} days")
        print(f"  Unit Price: ${rec['unit_price']:.2f}")
        print(f"  Stock Available: {rec['stock_available']:,} units")
        print(f"  Can Fulfill Order: {'Yes' if rec['can_fulfill'] else 'PARTIAL'}")
        
        print(f"\nFinancial Analysis:")
        print(f"  Price Premium: ${fin['price_premium']:,.0f}")
        print(f"  Avoided Loss: ${fin['avoided_loss']:,.0f}")
        print(f"  Net Benefit: ${fin['net_benefit']:,.0f}")
        
        print(f"\nRECOMMENDED ACTION:")
        print(f"  Initiate purchase requisition with {rec['vendor']}")
else:
    proposal = None
    print("\nNo mitigation proposal needed - risk level is acceptable.")

### Step 6.2 — The complete risk assessment dashboard

One view: prediction, tier, action level, order details, and the recommended mitigation.

In [ ]:
def display_risk_dashboard(po_row: pd.Series, predicted_delay: float, risk_tier: str,
                           action_level: str, proposal: Dict | None):
    """Display a formatted risk assessment dashboard."""
    
    tier_colors = {"Green": "#2e7d32", "Amber": "#f57f17", "Red": "#c62828"}
    action_colors = {"NORMAL": "#2e7d32", "ELEVATED": "#f57f17", 
                     "HIGH": "#e65100", "CRITICAL": "#c62828"}
    
    # Build alternatives table if available
    alt_table = ""
    if proposal and proposal.get("status") == "PROPOSAL_READY":
        rec = proposal["recommended_alternative"]
        fin = proposal["financial_impact"]
        
        alt_table = f"""
        <div style="margin-top:20px; padding:15px; background:#fff3e0; border-radius:6px;">
            <h4 style="margin-top:0; color:#e65100;">Recommended Mitigation</h4>
            <table style="width:100%;">
                <tr><td><strong>Alternative Vendor:</strong></td><td>{rec['vendor']} ({rec['country']})</td></tr>
                <tr><td><strong>Lead Time:</strong></td><td>{rec['lead_time_days']} days</td></tr>
                <tr><td><strong>Unit Price:</strong></td><td>${rec['unit_price']:.2f}</td></tr>
                <tr><td><strong>Stock Available:</strong></td><td>{rec['stock_available']:,} units</td></tr>
                <tr><td><strong>Price Premium:</strong></td><td>${fin['price_premium']:,.0f}</td></tr>
                <tr><td><strong>Avoided Loss:</strong></td><td style="color:#2e7d32; font-weight:bold;">${fin['avoided_loss']:,.0f}</td></tr>
                <tr><td><strong>Net Benefit:</strong></td><td style="font-weight:bold;">${fin['net_benefit']:,.0f}</td></tr>
            </table>
        </div>
        """
    
    dashboard_html = f"""
    <div style="font-family:system-ui,-apple-system,sans-serif; max-width:800px;">
        <div style="background:linear-gradient(135deg,#1a237e 0%,#283593 100%); color:white; 
                    padding:20px; border-radius:8px 8px 0 0;">
            <h2 style="margin:0;">JIT Risk Assessment Dashboard</h2>
            <p style="margin:5px 0 0 0; opacity:0.9;">PO: {po_row['PO_ID']} | Generated: {time.strftime('%Y-%m-%d %H:%M')}</p>
        </div>
        
        <div style="border:1px solid #e0e0e0; border-top:none; padding:20px; background:#fafafa;">
            <div style="display:grid; grid-template-columns:1fr 1fr 1fr; gap:20px; margin-bottom:20px;">
                <div style="background:white; padding:15px; border-radius:6px; text-align:center; 
                            border-left:4px solid {tier_colors.get(risk_tier, '#757575')};">
                    <div style="font-size:0.9em; color:#666;">Predicted Delay</div>
                    <div style="font-size:2em; font-weight:bold;">{predicted_delay} days</div>
                </div>
                <div style="background:white; padding:15px; border-radius:6px; text-align:center;
                            border-left:4px solid {tier_colors.get(risk_tier, '#757575')};">
                    <div style="font-size:0.9em; color:#666;">Risk Tier</div>
                    <div style="font-size:2em; font-weight:bold; color:{tier_colors.get(risk_tier, '#757575')};">{risk_tier}</div>
                </div>
                <div style="background:white; padding:15px; border-radius:6px; text-align:center;
                            border-left:4px solid {action_colors.get(action_level, '#757575')};">
                    <div style="font-size:0.9em; color:#666;">Action Level</div>
                    <div style="font-size:1.5em; font-weight:bold; color:{action_colors.get(action_level, '#757575')};">{action_level}</div>
                </div>
            </div>
            
            <div style="background:white; padding:15px; border-radius:6px; margin-bottom:15px;">
                <h4 style="margin-top:0;">Order Details</h4>
                <table style="width:100%;">
                    <tr><td style="width:30%;"><strong>Vendor:</strong></td><td>{po_row['Vendor_ID']} ({po_row['Vendor_Country']})</td></tr>
                    <tr><td><strong>Material:</strong></td><td>{po_row['Material_ID']} - {po_row['Material_Group']}</td></tr>
                    <tr><td><strong>Criticality:</strong></td><td style="color:{'#c62828' if po_row['Criticality_Flag']=='Yes' else '#666'};">
                        {'CRITICAL' if po_row['Criticality_Flag']=='Yes' else 'Standard'}</td></tr>
                    <tr><td><strong>Quantity:</strong></td><td>{int(po_row['Order_Quantity']):,} units</td></tr>
                    <tr><td><strong>Unit Price:</strong></td><td>${po_row['Net_Price']:.2f}</td></tr>
                    <tr><td><strong>Planned Lead Time:</strong></td><td>{int(po_row['Planned_Lead_Time_Days'])} days</td></tr>
                </table>
            </div>
            
            {alt_table}
        </div>
    </div>
    """
    
    display(HTML(dashboard_html))


# Display the dashboard
display_risk_dashboard(prediction_df.iloc[0], predicted_delay, risk_tier, action_level, proposal)

> ### ✓ CHECKPOINT — Section 6
> You built the full **Rules + Predict** pipeline: predict → classify → value → propose → *hand to a human*. The model never touched the ERP; it produced a recommendation a planner approves.
>
> 🏭 **What would break in production?** The approved action has to **write back** to S/4HANA safely — as a governed transaction with an audit trail, not a direct table write. *(That's a core Architecture Playbook topic.)*

---

## Section 7 — Apply to YOUR Landscape

📘 **KNOWLEDGE POINT — The ladder is the transferable skill**
The supply-chain use case is just a vehicle. The reusable architect's move is: *take a decision, place it on the Rules → Predict → Reason ladder, and name what you must govern at that altitude.* Do it now for a decision from **your own** world.

🎯 **Outcome:** you leave with your own use case mapped to the ladder — not just this one.

⌨️ **Your Turn:** edit the four answers in the cell below to describe a decision you actually own, then run it. The notebook maps your inputs onto the ladder and tells you what to govern.

In [ ]:
# ============================================================
# APPLY TO YOUR LANDSCAPE - fill in a decision from YOUR world
# ============================================================
# Pick a real decision (need not be supply chain). Edit the four
# answers, then run this cell to place it on the Decision Ladder.

MY_DECISION          = "e.g. Which incoming invoices to route for manual review"
DATA_IS_STRUCTURED   = True    # Mostly tabular / ERP data?                     (True/False)
HAS_HISTORICAL_LABEL = True    # Do you have past outcomes to learn from?        (True/False)
NEEDS_MULTI_STEP     = False   # Adaptive - next step depends on what you find?  (True/False)
NEEDS_HUMAN_SIGNOFF  = True    # Does acting require human approval?             (True/False)
# ============================================================

def recommend_rungs(structured, has_label, multi_step):
    rungs = ["Rung 1 - Rules"]                       # always the baseline
    if structured and has_label:
        rungs.append("Rung 2 - Predict (SAP-RPT-1)")
    if multi_step:
        rungs.append("Rung 3 - Reason (Agent on Gen AI Hub)")
    return rungs

govern = {
    "Rung 1 - Rules": "an audit trail of the logic",
    "Rung 2 - Predict (SAP-RPT-1)": "model accuracy + drift monitoring",
    "Rung 3 - Reason (Agent on Gen AI Hub)": "a reasoning trace + human approval",
}

rungs = recommend_rungs(DATA_IS_STRUCTURED, HAS_HISTORICAL_LABEL, NEEDS_MULTI_STEP)

print("YOUR DECISION:", MY_DECISION)
print("=" * 60)
print("Ladder altitude that fits your answers:")
for r in rungs:
    print(f"  {r:38} -> govern {govern[r]}")
print()
print(f"Highest rung to build: {rungs[-1]}")
if NEEDS_HUMAN_SIGNOFF:
    print("Acting needs human sign-off, so keep the top rung ADVISORY:")
    print("it recommends, a person approves - exactly like this workshop.")
print()
print("Rule of the ladder: climb only as high as the decision requires,")
print("and the higher you climb, the more you govern.")


> ### ✓ CHECKPOINT — Section 7
> If you can do this for one of your own decisions, you have the transferable skill this workshop exists to teach. Keep the printout — it's the seed of a real solution qualification.
>
> *We revisit this as a capstone worksheet after Exercise 2, once you've seen all three rungs run.*

---

## Section 8 — What You Built, and What Comes Next

You built the bottom two rungs of the Decision Ladder end to end:

| Rung | Component | What it did |
|------|-----------|-------------|
| 🔵 Rules | Vendor-OTIF tier + business policy | Cheap, auditable baseline — and you saw its blind spot |
| 🟢 Predict | **SAP-RPT-1** on SAP AI Core | Caught interaction risk the rule couldn't |
| 🔵+🟢 | Policy + mitigation proposal | Turned a prediction into a governed, human-approved action |

**The one idea to keep:** *climb only as high as the decision requires — and the higher you climb, the more you govern.*

---

> ### 🏗️ From Workshop to Production
>
> This exercise is **workshop-grade**. Taking the **Rules + Predict** rungs to production raises questions worth holding onto:
> - Where does prediction run — batch scoring, a real-time API, or event-driven on PO creation?
> - How do you monitor accuracy and detect drift once the model is live?
> - How does a risk score become a *governed* action inside your ERP?
>
> *→ After the hands-on, we'll work through these in an instructor-led architecture deep dive — the **Architecture Playbook** — on moving from workshop-grade to production.*


### What's next

- **Exercise 1B** *(optional)* — measure how good the prediction actually is, against open-source baselines.
- **Exercise 2** — climb to **Rung 3: Reason**, where an agent investigates multiple suppliers and reasons over trade-offs, still behind a human-approval gate.

*Continue to Exercise 1B (`exercise1b_jit_model_evaluation.ipynb`) or Exercise 2 (`exercise2_jit_agent.ipynb`).*

---

## Cleanup

Run at the end of the workshop to clear the cached OAuth token and sensitive environment variables from memory.

In [ ]:
def cleanup_session():
    """Clean up sensitive data at end of workshop."""
    global aicore_client
    
    print("Cleaning up workshop session...")
    
    # Clear AI Core client token
    if aicore_client is not None:
        aicore_client._access_token = None
        aicore_client._token_expires_at = 0.0
        print("   [OK] Cleared cached OAuth token")
    
    # Clear sensitive env vars from memory
    sensitive_vars = ["AICORE_CLIENT_ID", "AICORE_CLIENT_SECRET"]
    for var in sensitive_vars:
        if var in os.environ:
            del os.environ[var]
    print("   [OK] Cleared sensitive environment variables")
    
    print("\nCleanup complete.")

# Uncomment to run cleanup:
# cleanup_session()